In [ ]:
import pandas as pd
import requests
import time
from urllib.parse import quote

In [ ]:
# load the data
input_file = "animated_shows.csv"
output_file = "animated_shows_with_wiki.csv"

df = pd.read_csv(input_file, encoding='latin-1')
df

In [ ]:
# wikipedia settings
HEADERS = {
    "User-Agent": "KousalyaSimilarityProject/1.0 (educational project)"
}

In [ ]:
def fetch_wikipedia_summary(title):
    """
    Given a title, query Wikipedia's summary endpoint directly.
    Returns:
        wiki_title
        wiki_short_description
        wiki_summary
        wiki_url
        wiki_status
    """
    if pd.isna(title) or str(title).strip() == "":
        return None, None, None, None, "missing_title"

    clean_title = str(title).strip()
    encoded_title = quote(clean_title, safe="")

    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded_title}"

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)

        if response.status_code != 200:
            return None, None, None, None, f"failed_{response.status_code}"

        data = response.json()

        wiki_title = data.get("title")
        wiki_short_description = data.get("description")
        wiki_summary = data.get("extract")

        wiki_url = None
        if "content_urls" in data and "desktop" in data["content_urls"]:
            wiki_url = data["content_urls"]["desktop"].get("page")

        return wiki_title, wiki_short_description, wiki_summary, wiki_url, "success"

    except Exception as e:
        return None, None, None, None, f"error: {str(e)}"

In [ ]:
matched_titles = []
short_descriptions = []
summaries = []
status = []

In [ ]:
def fetch_wikipedia_summary(title):
    """
    Given a title, query Wikipedia's summary endpoint directly.
    Returns:
        wiki_title
        wiki_short_description
        wiki_summary
        wiki_url
        wiki_status
    """
    if pd.isna(title) or str(title).strip() == "":
        return None, None, None, None, "missing_title"

    clean_title = str(title).strip()
    encoded_title = quote(clean_title, safe="")

    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded_title}"

    try:
        response = requests.get(url, headers=HEADERS, timeout=15)

        if response.status_code != 200:
            return None, None, None, None, f"failed_{response.status_code}"

        data = response.json()

        wiki_title = data.get("title")
        wiki_short_description = data.get("description")
        wiki_summary = data.get("extract")

        wiki_url = None
        if "content_urls" in data and "desktop" in data["content_urls"]:
            wiki_url = data["content_urls"]["desktop"].get("page")

        return wiki_title, wiki_short_description, wiki_summary, wiki_url, "success"

    except Exception as e:
        return None, None, None, None, f"error: {str(e)}"

In [ ]:
wiki_titles = []
wiki_short_descriptions = []
wiki_summaries = []
wiki_urls = []
wiki_statuses = []

In [ ]:
for i, title in enumerate(df["Title"], start=1):
    wiki_title, short_desc, summary, wiki_url, status = fetch_wikipedia_summary(title)

    wiki_titles.append(wiki_title)
    wiki_short_descriptions.append(short_desc)
    wiki_summaries.append(summary)
    wiki_urls.append(wiki_url)
    wiki_statuses.append(status)

    # progress update every 50 rows
    if i % 50 == 0:
        print(f"Processed {i} rows...")

    # pause briefly so requests are polite
    time.sleep(0.2)

In [ ]:
# add new columns
df["wiki_title"] = wiki_titles
df["wiki_short_description"] = wiki_short_descriptions
df["wiki_summary"] = wiki_summaries
df["wiki_url"] = wiki_urls
df["wiki_status"] = wiki_statuses

In [ ]:
# save data
df.to_csv(output_file, index=False)

print("\nDone!")
print(f"Saved enriched file as: {output_file}")
print("\nStatus counts:")
print(df["wiki_status"].value_counts(dropna=False))

In [ ]:
print("Descriptions:", df["wiki_short_description"].notna().sum())
print("Summaries:", df["wiki_summary"].notna().sum())
print(df["wiki_status"].value_counts(dropna=False))